# UQRoute-TC — Qwen 2.5 1.5B APIBank and ToolAlpaca format audit

**Scope:** Twelve additional clean development cases from each benchmark, excluding the previously audited cases. The frozen single-sample settings are retained; held-out groups are excluded.

**Recorded evidence:** Saved raw responses, parser outcomes, official per-case scores, and chosen-token evidence for up to two parsed calls per benchmark.

## 1. Data and code revisions

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
root.mkdir(parents=True, exist_ok=True)
benchmark = Path('/content/robustbench-tc-release')
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
data_dir = benchmark / 'hf_data/datasets/api_eval'
assert not subprocess.check_output(['git', '-C', str(benchmark), 'status', '--porcelain', '--', str(data_dir)], text=True).strip(), 'Benchmark data has local changes: use a clean clone'

project = Path('/content/uqroute-tc-format-audit')
code_revision = 'bb4126441f849213dc7726b6319ac2a2e3e258cf'
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'feat/task2-canonical-parser'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', code_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == code_revision
print('Drive:', root)
print('Benchmark:', benchmark_revision)
print('Project code:', code_revision)


## 2. Inference environment

In [ ]:
import importlib.metadata
import json
import time
import urllib.request

try:
    installed_vllm = importlib.metadata.version('vllm')
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None
if installed_vllm != '0.30.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.30.0', 'openai>=1.50,<3', '-e', str(project)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.50,<3', '-e', str(project)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
import torch
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), 'Select an L4 GPU runtime'
assert importlib.metadata.version('vllm') == '0.30.0'

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
model_revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
endpoint = 'http://127.0.0.1:8000/v1'
server_log = Path('/content/uqroute_qwen15b_format_audit_server.log')
def served_models():
    try:
        with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
            return [item['id'] for item in json.load(response)['data']]
    except Exception:
        return None

active = served_models()
if active is not None and model_id not in active:
    raise RuntimeError(f'Port 8000 is serving another model: {active}')
if active and model_id in active:
    print('Server ready:', model_id)
else:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    if f'vllm serve {model_id}' in processes:
        print('Existing server is starting; waiting.')
        server = None
    else:
        with server_log.open('w') as log:
            server = subprocess.Popen(['vllm', 'serve', model_id, '--revision', model_revision,
                '--tokenizer-revision', model_revision, '--dtype', 'bfloat16',
                '--max-model-len', '8192', '--gpu-memory-utilization', '0.88',
                '--port', '8000'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print('Started server PID:', server.pid)
    for attempt in range(120):
        active = served_models()
        if active and model_id in active:
            print('Server ready:', model_id)
            break
        if server is not None and server.poll() is not None:
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
            raise RuntimeError('Server exited during startup')
        time.sleep(5)
    else:
        print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
        raise TimeoutError('Server was not ready within ten minutes')
print('GPU:', torch.cuda.get_device_name(0), '| vLLM:', importlib.metadata.version('vllm'))


## 3. Development case generation

In [ ]:
import hashlib
from openai import OpenAI
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import build_messages, load_samples, parse_tool_calls
from uqroute_tc.inference.pilot import run_cases

split_path = project / 'docs/splits/group_split_v1.json'
split = json.loads(split_path.read_text())
assert split['benchmark_revision'] == benchmark_revision
prior_dir = root / 'pilots/qwen25-15b-format-dev-four-v1'
prior = json.loads((prior_dir / 'manifest.json').read_text())
excluded_ids = set(prior['case_ids'])
wanted = ('apibank', 'toolalpaca')
selected = {name: [] for name in wanted}
for sample in load_samples(data_dir / 'clean.jsonl'):
    benchmark_name = sample['benchmark']
    if benchmark_name not in selected or str(sample['id']) in excluded_ids:
        continue
    assignment = split['assignments'].get(str(sample['id']))
    if assignment == {'benchmark': benchmark_name, 'population': 'primary', 'split': 'development'}:
        if len(selected[benchmark_name]) < 12:
            selected[benchmark_name].append(sample)
assert all(len(selected[name]) == 12 for name in wanted)
samples = [sample for name in wanted for sample in selected[name]]
print('Development cases:', {name: len(selected[name]) for name in wanted})

output_dir = root / 'pilots/qwen25-15b-xml-json-dev-24-v1'
config = {'benchmark_revision': benchmark_revision,
          'split_sha256': hashlib.sha256(split_path.read_bytes()).hexdigest(),
          'code_revision': code_revision, 'model_id': model_id,
          'model_revision': model_revision, 'tokenizer_revision': model_revision,
          'endpoint': endpoint, 'temperature': 0.001, 'max_tokens': 1024,
          'top_logprobs': 5, 'engine_version': '0.30.0', 'dtype': 'bfloat16',
          'max_model_len': 8192, 'mode': 'single_prompt_tool_calling',
          'selection': 'first 12 clean development rows per benchmark in pinned file order, excluding prior format audit',
          'excluded_case_ids': sorted(excluded_ids),
          'case_ids': [str(sample['id']) for sample in samples]}
assert model_id in (served_models() or []), 'Qwen 1.5B server unavailable'
client = OpenAI(base_url=endpoint, api_key='EMPTY', timeout=60, max_retries=0)
processed, resumed, failed = run_cases(samples, client, build_messages, parse_tool_calls, output_dir, config)
print(f'Collection: processed={processed} resumed={resumed} failed={failed}')
print('Records:', output_dir)


## 4. Parser audit and token evidence

In [ ]:
from collections import Counter
from google.colab import files
from uqroute_tc.inference.pilot import _record_path, token_evidence
from uqroute_tc.parsing.canonical import parse_prediction
sys.path.insert(0, str(benchmark / 'leaderboard/scoring'))
from eval_results import score_record

entries = []
alignment_evidence = []
candidate_count = {name: 0 for name in wanted}
for sample in samples:
    row = json.loads(_record_path(output_dir, str(sample['id'])).read_text())
    raw = row['prediction']['raw_output']
    parsed = parse_prediction(raw, sample['benchmark'], parse_tool_calls)
    response = row['uqroute']['response']
    choice = response['choices'][0] if response else None
    tokens = (choice.get('logprobs') or {}).get('content') or [] if choice else []
    evidence = token_evidence(choice) if choice else {'valid': False}
    result = score_record(row)
    entry = {
        'benchmark': sample['benchmark'], 'case_id': str(sample['id']),
        'record_status': row['uqroute']['status'],
        'parser_status': parsed.status, 'parsed_call_count': len(parsed.calls),
        'parsed_calls_match_saved': list(parsed.calls) == row['prediction']['tool_calls'],
        'token_evidence_valid': evidence['valid'],
        'finish_reason': choice.get('finish_reason') if choice else None,
        'correct': result['correct'], 'score': result['score'],
        'raw_preview': raw[:240],
    }
    entries.append(entry)
    if (row['uqroute']['status'] == 'complete' and parsed.status == 'calls'
            and entry['parsed_calls_match_saved'] and evidence['valid']
            and candidate_count[sample['benchmark']] < 2):
        alignment_evidence.append({
            'benchmark': sample['benchmark'], 'id': str(sample['id']),
            'raw_output': raw, 'parsed_calls': list(parsed.calls),
            'finish_reason': choice['finish_reason'],
            'chosen_tokens': tokens,
        })
        candidate_count[sample['benchmark']] += 1
        print('Selected:', sample['benchmark'], sample['id'])
        print('Raw:', repr(raw[:320]))

audit_path = output_dir / 'xml_json_format_audit.json'
audit_path.write_text(json.dumps({
    'manifest': config,
    'record_status_counts': dict(Counter(e['record_status'] for e in entries)),
    'parser_status_counts': {name: dict(Counter(e['parser_status'] for e in entries
        if e['benchmark'] == name)) for name in wanted},
    'selected_parseable_counts': candidate_count,
    'entries': entries,
}, ensure_ascii=False, indent=2) + '\n')
export_path = output_dir / 'xml_json_alignment_evidence.json'
export_path.write_text(json.dumps(alignment_evidence, ensure_ascii=False, indent=2) + '\n')
print('Parser statuses:', {name: dict(Counter(e['parser_status'] for e in entries
    if e['benchmark'] == name)) for name in wanted})
print('Selected parseable records:', candidate_count)
print('Audit:', audit_path)
print('Evidence:', export_path)
files.download(str(export_path))
